# Atenção! Este notebook demora consideravelmente para executar (15-25min).
## Os resultados pré-computados já estão disponíveis neste repositório, na pasta `plots`.
## O notebook não inclui o arquivo bônus com as animações.

### Estruturas base para o Modelo

In [1]:
# O objetivo desse arquivo é criar as estruturas básicas para as redes neurais do projetos, 
# como camadas, funções de ativação, otimizadores, etc.
from abc import ABC, abstractmethod
from mimetypes import init # Para criar classes abstratas. Não é importante para a lógica do código.
import numpy as np
SEED = 260382
np.random.seed(SEED) #Garantir experimentos reprodutíveis, usando a mesma seed para todas as inicializações aleatórias.
class Layer(ABC):
    @abstractmethod
    def forward(self, X):
        pass
    @abstractmethod
    def backward(self, grad):
        pass
    @abstractmethod
    def update_weights(self, learning_rate, Ridge=0, Lasso=0):
        pass
    @abstractmethod
    def clear_weights(self):
        pass

class Optimizer(ABC):
    def __init__(self, learning_rate, scheduler=None):
        self.learning_rate = learning_rate
        self.initial_learning_rate = learning_rate
        self.scheduler = scheduler

    @abstractmethod
    def update_weights(self, layers, Ridge=0, Lasso=0):
        pass

    def step_scheduler(self, epoch):
        if self.scheduler:
            self.learning_rate = self.scheduler.step(self.initial_learning_rate, epoch)

class LRScheduler(ABC):
    @abstractmethod
    def step(self, initial_lr, epoch):
        pass

class StepLR(LRScheduler):
    def __init__(self, step_size, gamma=0.1):
        self.step_size = step_size
        self.gamma = gamma

    def step(self, initial_lr, epoch):
        exponent = epoch // self.step_size
        return initial_lr * (self.gamma ** exponent)

class ExponentialLR(LRScheduler):
    def __init__(self, gamma=0.95):
        self.gamma = gamma

    def step(self, initial_lr, epoch):
        return initial_lr * (self.gamma ** epoch)

class LayerDense(Layer):
    def __init__(self, input_size, output_size, init = "Simple"):
        self.init = init
        self.input_size = input_size
        self.output_size = output_size
        self.init_weights(init)
        self.biases = np.zeros((1, output_size))
    def init_weights(self, init):
        if init == "Simple":
            self.weights = np.random.randn(self.input_size, self.output_size) * 2
        elif init == "Xavier":
            # Inicialização de Xavier (Glorot) - Ideal para ativações como Tanh ou Sigmoid
            # Variância = 1 / input_size
            desvio_padrao = np.sqrt(1.0 / self.input_size)
            self.weights = np.random.randn(self.input_size, self.output_size) * desvio_padrao
            
        elif init == "He":
            # Inicialização de He (Kaiming) - Ideal para ativações ReLU (O CASO DO SEU PROJETO!)
            # Variância = 2 / input_size
            desvio_padrao = np.sqrt(2.0 / self.input_size)
            self.weights = np.random.randn(self.input_size, self.output_size) * desvio_padrao
        else :
            raise NotImplementedError("Modo de inicialização não implementado.")
        
    def forward(self, X):
        self.input = X
        return np.dot(X, self.weights) + self.biases
    def backward(self, grad):
        self.grad_weights = np.dot(self.input.T, grad)
        self.grad_biases = np.sum(grad, axis=0, keepdims=True)
        return np.dot(grad, self.weights.T)
    def update_weights(self, learning_rate, Ridge=0, Lasso=0):
        self.weights -= learning_rate * self.grad_weights +  2 * Ridge * self.weights + Lasso * np.sign(self.weights)
        self.biases -= learning_rate * self.grad_biases
    def clear_weights(self):
        self.biases = np.zeros_like(self.biases)
        self.init_weights(self.init)

class Relu(Layer):
    def forward(self, X):
        self.input = X
        return np.maximum(0, X)
    def backward(self, grad):
        relu_grad = self.input > 0
        return grad * relu_grad
    def update_weights(self, learning_rate, Ridge=0, Lasso=0):
        pass # ReLU não tem pesos para atualizar
    def clear_weights(self):
        pass # ReLU não tem pesos para limpar
    
class SoftmaxCrossEntropy(Layer):
    #Combina a softmax e a cross-entropy em uma única camada para melhorar a estabilidade numérica.
    def __init__(self, Ridge = 0, Lasso = 0):
        super().__init__()
        self.Ridge = Ridge
        self.Lasso = Lasso
        
    def __forward(self, X, y, model=None):
        self.input = X
        self.y_true = y
        exp_scores = np.exp(X - np.max(X, axis=1, keepdims=True))
        self.probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        loss = -np.mean(np.log(self.probs[range(len(y)), y]))
        return loss
    def forward(self, X, y, model=None):
        loss = self.__forward(X, y, model)
        total_weights_abs = 0
        total_weights_sq = 0
        # Adicionar termos de regularização à perda
        if self.Ridge != 0:
            total_weights_sq = np.sum([np.sum(layer.weights ** 2) for layer in model.layers if hasattr(layer, 'weights')])  
            loss += self.Ridge * total_weights_sq
        if self.Lasso != 0:
            total_weights_abs = np.sum([np.sum(np.abs(layer.weights)) for layer in model.layers if hasattr(layer, 'weights')])  
            loss += self.Lasso * total_weights_abs
        
        return loss
    def backward(self, X, y, model=None):  
        loss = self.__forward(X, y, model)  # Recalcula a perda para garantir que as probabilidades estejam atualizadas
        grad = self.probs.copy()
        grad[range(len(y)), y] -= 1
        grad /= len(y)
        return grad
    def update_weights(self, learning_rate, Ridge=0, Lasso=0):
        pass # Camada de perda não tem pesos para atualizar  
    def clear_weights(self):        
        pass # Camada de perda não tem pesos para limpar
    
    
class FeatureExpansion(Layer):
    #Essa camada é responsável por expandir as características de entrada, criando novas características a partir das originais.
    #Como nosso trabalho é sobre fronteiras não lineares, essa camada pode ser bem útil para que o modelo consiga aprender essas fronteiras.
    def forward(self, X):
        self.input = X
        return np.hstack((X, X**2))
    def backward(self, grad):
        grad_input = grad[:, :self.input.shape[1]] + 2 * self.input * grad[:, self.input.shape[1]:2*self.input.shape[1]]
        return grad_input
    def update_weights(self, learning_rate, Ridge=0, Lasso=0):
        pass # Camada de expansão de características não tem pesos para atualizar
    def clear_weights(self):        
        pass # Camada de expansão de características não tem pesos para limpar
    

class SGD(Optimizer):
    def __init__(self, learning_rate, scheduler=None):
        super().__init__(learning_rate, scheduler)
    def update_weights(self, layers, Ridge=0, Lasso=0):
        for layer in layers:
            layer.update_weights(self.learning_rate, Ridge, Lasso)

class ADAM(Optimizer):
    # Extra mencionado no pdf, interessante de se fazer depois para melhorar o desempenho do modelo.
    def __init__(self, learning_rate, beta1=0.9, beta2=0.999, epsilon=1e-8, scheduler=None):
        super().__init__(learning_rate, scheduler)
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}
        self.v = {}
        self.t = 0

    def update_weights(self, layers, Ridge=0, Lasso=0):
        self.t += 1
        beta1_t = 1 - self.beta1 ** self.t
        beta2_t = 1 - self.beta2 ** self.t

        for i, layer in enumerate(layers):
            if hasattr(layer, 'weights') and hasattr(layer, 'grad_weights'):
                if i not in self.m:
                    self.m[i] = {
                        'w': np.zeros_like(layer.weights),
                        'b': np.zeros_like(layer.biases)
                    }
                    self.v[i] = {
                        'w': np.zeros_like(layer.weights),
                        'b': np.zeros_like(layer.biases)
                    }

                grad_w = layer.grad_weights + 2 * Ridge * layer.weights + Lasso * np.sign(layer.weights)
                grad_b = layer.grad_biases

                self.m[i]['w'] = self.beta1 * self.m[i]['w'] + (1 - self.beta1) * grad_w
                self.m[i]['b'] = self.beta1 * self.m[i]['b'] + (1 - self.beta1) * grad_b

                self.v[i]['w'] = self.beta2 * self.v[i]['w'] + (1 - self.beta2) * (grad_w ** 2)
                self.v[i]['b'] = self.beta2 * self.v[i]['b'] + (1 - self.beta2) * (grad_b ** 2)

                m_hat_w = self.m[i]['w'] / beta1_t
                m_hat_b = self.m[i]['b'] / beta1_t
                v_hat_w = self.v[i]['w'] / beta2_t
                v_hat_b = self.v[i]['b'] / beta2_t

                layer.weights -= self.learning_rate * m_hat_w / (np.sqrt(v_hat_w) + self.epsilon)
                layer.biases -= self.learning_rate * m_hat_b / (np.sqrt(v_hat_b) + self.epsilon)
    
class Model:
    def __init__(self, name: str, layers = [], loss = None, optimizer = None):
        self.name = name
        self.layers = layers
        self.loss = loss
        self.optimizer = optimizer
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    def backward_gradient(self, y_pred, y_true):
        grad = self.loss.backward(y_pred, y_true, self)
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad
    def update_weights(self, ):
        self.optimizer.update_weights(self.layers, self.loss.Ridge, self.loss.Lasso)
    def train(self, X_train, y_train, X_test=None, y_test=None, epochs = None, batch_size = None):
        train_losses = []
        train_accs = []
        test_losses = [] if X_test is not None and y_test is not None else None
        test_accs = [] if X_test is not None and y_test is not None else None

        # Função auxiliar para calcular acurácia
        def calc_accuracy(y_pred, y_true):
            predictions = np.argmax(y_pred, axis=1)
            return np.mean(predictions == y_true)

        y_pred = self.forward(X_train)
        train_losses.append(self.loss.forward(y_pred, y_train, self))
        train_accs.append(calc_accuracy(y_pred, y_train))
        if test_losses is not None:
            y_test_pred = self.forward(X_test)
            test_losses.append(self.loss.forward(y_test_pred, y_test, self))
            test_accs.append(calc_accuracy(y_test_pred, y_test))
        
        for epoch in range(1, epochs + 1):
            for i in range(0, len(X_train), batch_size):
                X_batch = X_train[i:i+batch_size]
                y_batch = y_train[i:i+batch_size]
                
                y_pred = self.forward(X_batch)
                loss = self.loss.forward(y_pred, y_batch, self)
                grad = self.backward_gradient(y_pred, y_batch)
                self.update_weights()
            
            self.optimizer.step_scheduler(epoch)

            y_pred = self.forward(X_train)
            train_losses.append(self.loss.forward(y_pred, y_train, self))
            train_accs.append(calc_accuracy(y_pred, y_train))
            if test_losses is not None:
                y_test_pred = self.forward(X_test)
                test_losses.append(self.loss.forward(y_test_pred, y_test, self))
                test_accs.append(calc_accuracy(y_test_pred, y_test))
        return train_losses, test_losses, train_accs, test_accs
    def clear(self):
        for layer in self.layers:
            layer.clear_weights()

        self.optimizer.learning_rate = self.optimizer.initial_learning_rate

        if hasattr(self.optimizer, 'velocities'):
            self.optimizer.velocities = {}
            
        if hasattr(self.optimizer, 'm'):
            self.optimizer.m = {}
            self.optimizer.v = {}
            self.optimizer.t = 0

class SGDMomentum(Optimizer):
    def __init__(self, learning_rate, beta=0.9, scheduler=None):
        super().__init__(learning_rate, scheduler)
        self.beta = beta
        # Dicionário para guardar as "velocidades" de cada camada pelo índice
        self.velocities = {}
        
    def update_weights(self, layers, Ridge=0, Lasso=0):
        for i, layer in enumerate(layers):
            # Apenas camadas com pesos precisam ser atualizadas
            if hasattr(layer, 'weights') and hasattr(layer, 'grad_weights'):
                # Inicializa as velocidades com zero na primeira iteração
                if i not in self.velocities:
                    self.velocities[i] = {
                        'w': np.zeros_like(layer.weights),
                        'b': np.zeros_like(layer.biases)
                    }
                
                # Resgata o gradiente atual e aplica as regularizações (L1/L2) já implementadas
                grad_w = layer.grad_weights + 2 * Ridge * layer.weights + Lasso * np.sign(layer.weights)
                grad_b = layer.grad_biases
                
                # 1. Calcula a nova velocidade (Momento)
                self.velocities[i]['w'] = self.beta * self.velocities[i]['w'] + (1 - self.beta) * grad_w
                self.velocities[i]['b'] = self.beta * self.velocities[i]['b'] + (1 - self.beta) * grad_b
                
                # 2. Atualiza os pesos reais da camada
                layer.weights -= self.learning_rate * self.velocities[i]['w']
                layer.biases -= self.learning_rate * self.velocities[i]['b']




### Geração de Datasets

In [2]:
import numpy as np
from sklearn.datasets import make_circles, make_moons
from sklearn.model_selection import train_test_split
import platform
from PIL import Image, ImageDraw, ImageFont

def spiral_2d(n_samples, noise, rotations=2.0, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples_per_class = n_samples // 2
    X = np.zeros((n_samples, 2))
    y = np.zeros(n_samples, dtype=int)
    
    # Raiz quadrada suaviza a densidade, evitando um "borrão" concentrado na origem
    base_t = np.sqrt(np.random.rand(n_samples_per_class)) * (rotations * 2 * np.pi)
    
    for j in range(2):
        ix = range(n_samples_per_class * j, n_samples_per_class * (j + 1))
        
        # O raio cresce proporcionalmente ao ângulo
        r = base_t / (rotations * 2 * np.pi)
        
        # O pulo do gato: A Classe 0 tem defasagem 0. A Classe 1 tem defasagem de PI.
        t = base_t + (j * np.pi)
        
        # Conversão polar para cartesiana + Ruído espacial isotrópico (Gaussian)
        X[ix, 0] = r * np.cos(t) + np.random.randn(n_samples_per_class) * noise
        X[ix, 1] = r * np.sin(t) + np.random.randn(n_samples_per_class) * noise
        y[ix] = j
        
    return X, y


def text_2d(text_line1="MC", text_line2="906", n_samples=500, noise=0.01, random_state=None, width=800, height=800):
    """
    Gera um dataset com texto em preto sobre fundo branco de forma vetorizada.
    
    Classe 0: Fundo branco
    Classe 1: Letras em preto
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    img = Image.new('L', (width, height), color=255)
    draw = ImageDraw.Draw(img)
    
    # Gerenciamento robusto de fontes cross-platform
    try:
        os_name = platform.system()
        if os_name == "Windows":
            font = ImageFont.truetype("arial.ttf", 400)
        elif os_name == "Darwin": # macOS
            font = ImageFont.truetype("Arial.ttf", 400)
        else: # Linux
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 400)
    except OSError:
        print("AVISO CRÍTICO: Fonte TrueType não encontrada. O texto usará o fallback e ficará ilegível.")
        font = ImageFont.load_default()
    
    # Desenhar primeira linha
    bbox1 = draw.textbbox((0, 0), text_line1, font=font)
    x1 = (width - (bbox1[2] - bbox1[0])) // 2
    y1 = height // 4 - (bbox1[3] - bbox1[1]) // 2
    y1 = max(0, min(y1, height - (bbox1[3] - bbox1[1])))
    draw.text((x1, y1), text_line1, fill=0, font=font)
    
    # Desenhar segunda linha
    bbox2 = draw.textbbox((0, 0), text_line2, font=font)
    x2 = (width - (bbox2[2] - bbox2[0])) // 2
    y2 = height * 3 // 4 - (bbox2[3] - bbox2[1]) // 2 - 50
    y2 = max(0, min(y2, height - (bbox2[3] - bbox2[1]) - 50))
    draw.text((x2, y2), text_line2, fill=0, font=font)
    
    img_array = np.array(img)
    
    # VETORIZAÇÃO: Criar matrizes de coordenadas X e Y
    j, i = np.meshgrid(np.arange(width), np.arange(height))
    
    # Normalização isotrópica (usando o mesmo fator de escala para manter o aspect ratio 4:1)
    scale = width / 2
    x_norm = (j - width / 2) / scale
    
    # Correção crucial: Inverter o sinal do eixo Y para compatibilidade cartesiana
    y_norm = -(i - height / 2) / scale 
    
    # Separação por máscaras booleanas (Thresholding vetorizado)
    mask_0 = img_array > 127
    mask_1 = img_array <= 127
    
    class_0_points = np.column_stack((x_norm[mask_0], y_norm[mask_0]))
    class_1_points = np.column_stack((x_norm[mask_1], y_norm[mask_1]))
    
    # Determinar número de amostras
    if n_samples is not None:
        samples_per_class = n_samples // 2
    else:
        samples_per_class = min(len(class_0_points), len(class_1_points))
        
    samples_per_class = min(samples_per_class, len(class_0_points), len(class_1_points))
    
    # Amostragem (sem reposição)
    indices_0 = np.random.choice(len(class_0_points), samples_per_class, replace=False)
    indices_1 = np.random.choice(len(class_1_points), samples_per_class, replace=False)
    
    X_0 = class_0_points[indices_0]
    X_1 = class_1_points[indices_1]
    
    # Adicionar ruído
    if noise > 0:
        X_0 += np.random.randn(*X_0.shape) * noise
        X_1 += np.random.randn(*X_1.shape) * noise
        
    # Combinação e embaralhamento final
    X = np.vstack([X_0, X_1])
    y_labels = np.hstack([np.zeros(samples_per_class, dtype=int), 
                          np.ones(samples_per_class, dtype=int)])
    
    permutation = np.random.permutation(len(X))
    return X[permutation], y_labels[permutation]


def generate_datasets():
    data_sets = []
    X_circles, Y_Circles = make_circles(n_samples= 200, noise=0.2, factor=0.3, random_state= 260382) #Seed  para garantir resultados consistentes(RA de um membro do grupo)
    X_train_circles, X_test_circles, Y_train_circles, Y_test_circles = train_test_split(
        X_circles, Y_Circles, test_size=0.2, random_state=260382
    )
    data_sets.append((X_train_circles, Y_train_circles, X_test_circles, Y_test_circles, "Circles"))
    
    X_moons, Y_Moons = make_moons(n_samples= 200, noise=0.1, random_state= 260382) #Seed para garantir resultados consistentes(RA de um membro do grupo)
    X_train_moons, X_test_moons, Y_train_moons, Y_test_moons = train_test_split(
        X_moons, Y_Moons, test_size=0.2, random_state=260382
    )
    data_sets.append((X_train_moons, Y_train_moons, X_test_moons, Y_test_moons, "Moons"))
    
    X_spiral, Y_Spiral = spiral_2d(n_samples=500, noise=0.01, random_state=260382) #Seed para garantir resultados consistentes(RA de um membro do grupo)
    X_train_spiral, X_test_spiral, Y_train_spiral, Y_test_spiral = train_test_split(
        X_spiral, Y_Spiral, test_size=0.2, random_state=260382
    )
    data_sets.append((X_train_spiral, Y_train_spiral, X_test_spiral, Y_test_spiral, "Spiral"))
    
    X_text, Y_Text = text_2d( n_samples=2000, noise=0.00, random_state=260382) #Seed para garantir resultados consistentes(RA de um membro do grupo)
    X_train_text, X_test_text, Y_train_text, Y_test_text = train_test_split(
        X_text, Y_Text, test_size=0.05, random_state=260382
    )
    data_sets.append((X_train_text, Y_train_text, X_test_text, Y_test_text, "MC906"))
    return data_sets

### Biblioteca de plots


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

PLOTS_DIR = "plots"

def plot_decision_boundary(model: Model, X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray = None, y_test: np.ndarray = None, data_name: str = ""):
    name = model.name
    X_all = X_train if X_test is None else np.vstack((X_train, X_test))
    y_all = y_train if y_test is None else np.concatenate((y_train, y_test))
    h = 0.01
    x_min, x_max = X_all[:, 0].min() - 0.25, X_all[:, 0].max() + 0.25
    y_min, y_max = X_all[:, 1].min() - 0.25, X_all[:, 1].max() + 0.25
    
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    logits = model.forward(grid_points)
    
    if logits.ndim == 2 and logits.shape[1] == 2:
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exp_logits[:, 1] / np.sum(exp_logits, axis=1)
    else:
        probs = 1 / (1 + np.exp(-logits.flatten()))
        
    Z = probs.reshape(xx.shape)
    cmap_custom = LinearSegmentedColormap.from_list("BlueOrange", ["#4B8BBE", "#FCEFDD", "#F29D4B"])
    
    plt.figure(figsize=(8, 8))
    plt.contourf(xx, yy, Z, levels=100, cmap=cmap_custom, alpha=0.9, vmin=0, vmax=1)
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cmap_custom, edgecolors='white', linewidth=1, s=40, label='Train')
    if X_test is not None and y_test is not None:
        plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cmap_custom, marker='X', edgecolors='black', linewidth=1.2, s=60, label='Test')
    plt.title(f"Fronteira de Decisão Modelo {name}")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.legend()
    
    SAVE_DIR = os.path.join(PLOTS_DIR, data_name, name)
    os.makedirs(SAVE_DIR, exist_ok=True)
    plt.savefig(os.path.join(SAVE_DIR,f"Fronteira_{name}.png"), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
def plot_internal_decision_boundries(model: Model, X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray = None, y_test: np.ndarray = None, data_name: str = ""):
    # Plot a fronteira de decisão interna do modelo, mostrando a saída de cada neurônio em cada camada.
    X = X_train if X_test is None else np.vstack((X_train, X_test))
    #y = y_train if y_test is None else np.concatenate((y_train, y_test))

    x_min, x_max = X[:, 0].min() - 0.25, X[:, 0].max() + 0.25
    y_min, y_max = X[:, 1].min() - 0.25, X[:, 1].max() + 0.25
    span = max(x_max - x_min, y_max - y_min)
    x_mid = 0.5 * (x_min + x_max)
    y_mid = 0.5 * (y_min + y_max)
    x_min, x_max = x_mid - span / 2, x_mid + span / 2
    y_min, y_max = y_mid - span / 2, y_mid + span / 2
    h = 0.02
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]

    activations = [grid]
    layer_names = ['Input']
    out = grid
    for layer in model.layers:
        out = layer.forward(out)
        activations.append(out if out.ndim == 2 else out.reshape(-1, 1))
        layer_names.append(layer.__class__.__name__)

    soft = out
    if soft.ndim == 2 and soft.shape[1] == 2:
        exp_scores = np.exp(soft - np.max(soft, axis=1, keepdims=True))
        soft = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    else:
        soft = 1 / (1 + np.exp(-soft.flatten()))
        soft = soft.reshape(-1, 1)
    activations.append(soft)
    layer_names.append('Softmax')

    n_layers = len(activations)
    max_neurons = max(act.shape[1] for act in activations)
    square = 3.0
    width = max(16, square * n_layers)
    height = max(9, square * max_neurons)
    fig, axes = plt.subplots(max_neurons, n_layers, figsize=(width, height), squeeze=False)
    fig.subplots_adjust(top=0.88, hspace=0.15, wspace=0.16)
    cmap_custom = LinearSegmentedColormap.from_list("BlueOrange", ["#4B8BBE", "#FCEFDD", "#F29D4B"])

    for col, act in enumerate(activations):
        neurons = act.shape[1]
        start = (max_neurons - neurons) // 2
        

        for row in range(max_neurons):
            ax = axes[row][col]
            if row < start or row >= start + neurons:
                ax.axis('off')
                continue
            Z = act[:, row - start].reshape(xx.shape)
            ax.contourf(xx, yy, Z, levels=100, cmap=cmap_custom, alpha=0.9, vmin=-1.0, vmax=1.0)
            #ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_custom, edgecolors='white', s=16, linewidth=0.3, alpha=0.8)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            ax.set_aspect('equal', adjustable='box')

        header_ax = axes[start][col]
        header_ax.set_title(f'Camada {col + 1}: {layer_names[col]}', pad=12, fontsize=10)

    for col in range(n_layers):
        for row in range(max_neurons):
            if axes[row][col].lines or axes[row][col].collections:
                break
            axes[row][col].axis('off')

    plt.suptitle(f'Fronteiras Internas por Neurônio — Modelo {model.name}', fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.91])

    SAVE_DIR = os.path.join(PLOTS_DIR, data_name, model.name)
    os.makedirs(SAVE_DIR, exist_ok=True)
    plt.savefig(os.path.join(SAVE_DIR, f'InternalBoundaries_{model.name}.png'), dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

def plot_loss_curve(model, train_losses, test_losses=None, data_name: str = ""):
    #Plota a curva de perda ao longo do treinamento, tanto para os dados de treino quanto para os de teste (se fornecidos).
    #Escala logaritmica no eixo das épcoas para melhor visualização, já que a perda pode diminuir rapidamente no início do treinamento.
    plt.plot(train_losses, label='Train')
    if test_losses is not None:
        plt.plot(test_losses, label='Test')
    plt.xscale('log')
    plt.ylim(bottom=0, top=1.0)
    name = model.name
    # Habilita os marcadores menores (subdivisões)
    plt.minorticks_on()
    
    # Grid principal (linhas sólidas, mais escuras)
    plt.grid(which='major', color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    
    # Grid secundário (linhas pontilhadas, mais claras)
    plt.grid(which='minor', color='gray', linestyle=':', linewidth=0.5, alpha=0.5)
    
    plt.title(f"Perda ao longo do treinamento - Modelo {name}")
    plt.xlabel("Época")
    plt.ylabel("Perda")
    plt.legend()
    
    SAVE_DIR = os.path.join(PLOTS_DIR, data_name, name)
    os.makedirs(SAVE_DIR, exist_ok=True)
    plt.savefig(os.path.join(SAVE_DIR,f"Losses_{name}.png"), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

def plot_accuracy_curve(model, train_accs, test_accs=None, data_name: str = ""):
    # Plota a curva de acurácia ao longo do treinamento
    plt.plot(train_accs, label='Train')
    if test_accs is not None:
        plt.plot(test_accs, label='Test')
        
    # A escala para acurácia é linear, de 0 a 1 (0% a 100%)
    plt.ylim(bottom=0.0, top=1.05)
    
    name = model.name
    # Habilita os marcadores menores (subdivisões)
    plt.minorticks_on()
    
    # Grid principal e secundário
    plt.grid(which='major', color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    plt.grid(which='minor', color='gray', linestyle=':', linewidth=0.5, alpha=0.5)
    
    plt.title(f"Acurácia ao longo do treinamento - Modelo {name}")
    plt.xlabel("Época")
    plt.ylabel("Acurácia")
    plt.legend()
    
    SAVE_DIR = os.path.join(PLOTS_DIR, data_name, name)
    os.makedirs(SAVE_DIR, exist_ok=True)
    plt.savefig(os.path.join(SAVE_DIR, f"Accuracy_{name}.png"), dpi=300, bbox_inches='tight')

    # Salva um resumo em texto com as últimas acurácias de treino e teste
    try:
        last_train = float(train_accs[-1]) if len(train_accs) > 0 else float('nan')
    except Exception:
        last_train = float('nan')

    if test_accs is not None and len(test_accs) > 0:
        try:
            last_test = float(test_accs[-1])
        except Exception:
            last_test = float('nan')
    else:
        last_test = None

    summary_path = os.path.join(SAVE_DIR, f'Accuracy_summary_{name}.txt')
    with open(summary_path, 'w') as f:
        f.write(f'Train_last_accuracy: {last_train:.6f}\n')
        if last_test is None:
            f.write('Test_last_accuracy: N/A\n')
        else:
            f.write(f'Test_last_accuracy: {last_test:.6f}\n')

    plt.show()
    plt.close()


def softmax_probs(logits):
    """Estabiliza e calcula a probabilidade (mesma lógica da sua Loss)."""
    exp_scores = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

def plot_neuron_ablation_tv(model, X, y=None, layer_index=None, neuron_index=None, dataset_name=None):
    """
    Gera um Heatmap da TV Distance mostrando a região de responsabilidade 
    de um neurônio específico, sem modificar a estrutura do modelo.
    """
    # 1. Definir os limites do Grid com base nos dados (similar ao seu plot_decision_boundary)
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                         np.arange(y_min, y_max, 0.05))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # 2. Obter as probabilidades do modelo INTACTO (Baseline)
    logits_orig = model.forward(grid)
    P_orig = softmax_probs(logits_orig) # Shape: (N_pontos, 2 classes)

    # 3. Acessar a camada alvo (precisa ser uma LayerDense)
    target_layer = model.layers[layer_index]
    if not hasattr(target_layer, 'weights'):
        raise ValueError("A camada especificada não possui pesos (não é LayerDense).")

    # 4. CIRURGIA DE ABLAÇÃO (Backup -> Zerar -> Prever -> Restaurar)
    # Fazer backup físico (copy) para não perder os ponteiros
    w_backup = np.copy(target_layer.weights[:, neuron_index])
    b_backup = np.copy(target_layer.biases[0, neuron_index])

    try:
        # Forçar a morte do neurônio
        target_layer.weights[:, neuron_index] = 0
        target_layer.biases[0, neuron_index] = 0

        # Obter probabilidades do modelo ABLATADO
        logits_abl = model.forward(grid)
        P_abl = softmax_probs(logits_abl)

    finally:
        # Garantir a restauração incondicional dos pesos originais
        target_layer.weights[:, neuron_index] = w_backup
        target_layer.biases[0, neuron_index] = b_backup

    # 5. Calcular a Distância de Variação Total (TV Distance)
    # TV = 1/2 * soma_sobre_classes( | P_orig - P_abl | )
    D_TV = 0.5 * np.sum(np.abs(P_orig - P_abl), axis=1)
    D_TV = D_TV.reshape(xx.shape)

    # 6. Plotar o Heatmap
    plt.figure(figsize=(8, 6))
    # Usamos o colormap 'Reds'. Branco = 0 deformação, Vermelho Escuro = alta deformação 
    contour = plt.contourf(xx, yy, D_TV, levels=20, cmap='Reds', vmin=0.0, vmax=1.0)
    plt.colorbar(contour, label='TV Distance (Degradação da Probabilidade)')
    
    # Plotar o dataset em background com transparência para dar contexto espacial
    # Se labels (`y`) forem fornecidas, colorimos por classe; caso contrário,
    # usamos as predições do modelo intacto para inferir a classe.
    if y is None:
        inferred_y = np.argmax(P_orig, axis=1)
    else:
        inferred_y = np.ravel(y)

    # Usar azul para a classe 0 e laranja para a classe 1
    class_cmap = LinearSegmentedColormap.from_list("BlueOrangeClasses", ["#4B8BBE", "#F29D4B"])
    plt.scatter(X[:, 0], X[:, 1], c=inferred_y, cmap=class_cmap, edgecolors='k', alpha=0.3, s=20)
    
    plt.title(f'Ablação: Camada {layer_index}, Neurônio {neuron_index}\n{model.name} - {dataset_name}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    
    SAVE_DIR = os.path.join(PLOTS_DIR, dataset_name, model.name, "Ablation")
    os.makedirs(SAVE_DIR, exist_ok=True)
    plt.savefig(os.path.join(SAVE_DIR, f'Ablação:Camada_{layer_index}_N{neuron_index}_{model.name}.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [4]:

#Esse arquivo faz exatamente a mesma coisa que main, mas não mostra os gráficos na tela, apenas salva os arquivos.
# Ele é útil para gerar os gráficos sem precisar fichar fechando as janelas dos gráficos a cada execução.
import matplotlib
matplotlib.use('Agg')  # Use o backend 'Agg' para evitar problemas de exibição em ambientes sem suporte gráfico

n = "u"
while n not in ["S", "N"]:
    n = input("Gostaria de Salvar a fronteiras internas? S/N: ").strip()
PLOT_INTERNAL_BOUNDARIES = {"S": True, "N":False}[n]


### Main onde o treino e avaliação principal ocorrem.

In [5]:
import numpy as np
import matplotlib.pyplot as plt



def test_model(model:Model, data_sets):
    global PLOT_INTERNAL_BOUNDRIES
    for X_train, Y_train, X_test, Y_test, dataset_name in data_sets:
        model.clear()
        train_losses, test_losses, train_accs, test_accs = model.train(
            X_train,
            Y_train,
            X_test,
            Y_test,
            epochs=3000,
            batch_size=32,
        )
        plot_loss_curve(model, train_losses, test_losses, dataset_name)
        plot_accuracy_curve(model, train_accs, test_accs, dataset_name)
        plot_decision_boundary(model, X_train, Y_train, X_test, Y_test, dataset_name)
        if (PLOT_INTERNAL_BOUNDARIES):
            plot_internal_decision_boundries(model, X_train, Y_train, X_test, Y_test, dataset_name)
            
            for layer_index, layer in enumerate(model.layers):
                # A ablação só faz sentido em camadas que possuem pesos e vieses (LayerDense)
                if hasattr(layer, 'weights'):
                    
                    # Descomente o bloco abaixo se NÃO quiser ablatar a última camada (os 2 neurônios de saída das classes).
                    # Ablatar a última camada geralmente não traz informações sobre representação interna.
                    # if layer_index == len(model.layers) - 1:
                    #     continue
                    
                    num_neurons = layer.output_size
                    for neuron_index in range(num_neurons):
                        print(f"  -> Processando e salvando: Camada {layer_index}, Neurônio {neuron_index}/{num_neurons-1}")
                        
                        # Certifique-se de que a função abaixo esteja importada no início do main.py
                        # (ex: from plot_utils import plot_neuron_ablation_tv)
                        plot_neuron_ablation_tv(
                            model=model, 
                            X=X_train, # Usamos o X_train para definir os limites do grid 2D
                            y=Y_train,
                            layer_index=layer_index, 
                            neuron_index=neuron_index, 
                            dataset_name=dataset_name
                        )
        else:
            print("Plot Internal Boundary Disabled")
    
def define_models():
    models = []

    modelo_simples = Model(
        "Modelo_Simples",
        [LayerDense(2, 6, init="He"), Relu() ,LayerDense(6, 2, init="He")],
        SoftmaxCrossEntropy(),
        SGD(learning_rate=0.1)
    )
    # #ATENÇÃO, o retorno do modelo está em logits, então a função de perda já inclui a softmax.
    # #O resultados podem não estar entre 0 e 1.
    # models.append(modelo_simples)
    models.append(modelo_simples)

    modelo_expansivo = Model(
        "Modelo_Expansivo",
        [FeatureExpansion(), LayerDense(4, 6, init="He"), Relu(), LayerDense(6, 2, init="He")],
        SoftmaxCrossEntropy(Ridge=2e-6),
        SGD(learning_rate=0.1)
    )
    models.append(modelo_expansivo)

    modelo_momentum = Model(
        "Modelo_Momentum", 
        [LayerDense(2, 6), Relu(), LayerDense(6, 2)], 
        SoftmaxCrossEntropy(), 
        SGDMomentum(learning_rate=0.1, beta=0.9, scheduler=StepLR(step_size=500, gamma=1.0))
    )
    models.append(modelo_momentum)

    return models

def define_experimental_models():
    models = []

    modelo_wide_shallow = Model("Modelo_Wide_Shallow1x20",
    [
        LayerDense(2, 20, init="He"), 
        Relu(), 
        LayerDense(20, 20, init="He"), 
        Relu(), 
        LayerDense(20, 2, init="He")
    ],
    SoftmaxCrossEntropy(), ADAM(learning_rate=0.001, scheduler=StepLR(step_size=500, gamma=0.5)))
    models.append(modelo_wide_shallow)

    modelo_balanced = Model("Modelo_Balanced2x10",
    [
        LayerDense(2, 10, init="He"), 
        Relu(), 
        LayerDense(10, 10, init="He"), 
        Relu(),
        LayerDense(10, 10, init="He"), 
        Relu(), 
        LayerDense(10, 2, init="He")
    ],
    SoftmaxCrossEntropy(Ridge=0.01), ADAM(learning_rate=0.001, scheduler=StepLR(step_size=500, gamma=0.5)))
    models.append(modelo_balanced)

    modelo_narrow_deep = Model("Modelo_Narrow_Deep4x5",
    [
        LayerDense(2, 5, init="He"), 
        Relu(), 
        LayerDense(5, 5, init="He"), 
        Relu(),
        LayerDense(5, 5, init="He"), 
        Relu(), 
        LayerDense(5, 5, init="He"), 
        Relu(), 
        LayerDense(5, 5, init="He"), 
        Relu(), 
        LayerDense(5, 2, init="He")
    ],
    SoftmaxCrossEntropy(Ridge=0.001), ADAM(learning_rate=0.001, scheduler=StepLR(step_size=500, gamma=0.5)))
    models.append(modelo_narrow_deep)

    return models

def main():
   
    models = define_models() + define_experimental_models()
    full_data_sets = generate_datasets()
    for model in models:
        print(f"Treinando e avaliando o modelo: {model.name}")
        test_model(model, full_data_sets)
if __name__ == "__main__":
    main()

Treinando e avaliando o modelo: Modelo_Simples


/tmp/ipykernel_16109/1226199996.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16109/1226199996.py:206: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_16109/1226199996.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Treinando e avaliando o modelo: Modelo_Expansivo
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Treinando e avaliando o modelo: Modelo_Momentum
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Treinando e avaliando o modelo: Modelo_Wide_Shallow1x20
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Treinando e avaliando o modelo: Modelo_Balanced2x10
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Treinando e avaliando o modelo: Modelo_Narrow_Deep4x5
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot Internal Boundary Disabled
Plot 